# The Synthetic Control Ladder in Python

## A guided tour of `mlsynth` on the Brexit referendum

Companion notebook for [carlos-mendez.org/post/python_sc_dsc_sdid](https://carlos-mendez.org/post/python_sc_dsc_sdid/)

We climb the whole ladder of single-treated-unit estimators — DiD → SC → DSC → SDID → MASC → ASCM — with **one `mlsynth` class per rung**, on quarterly log real GDP for 24 OECD economies (1995Q1–2020Q4). The United Kingdom is the treated unit, 23 countries are donors, and the referendum is dated 2016Q3.

The focus is the *package*: which config field selects which estimator, what the result object contains, and where a default will quietly hand you a different estimator than the one you meant to fit.

> **Naming hazard, before you import anything.** `mlsynth` ships a class called `DSC`. It is **not** the estimator in this notebook.
> - `mlsynth.DSC` = **Distributional** synthetic control (Gunsilius 2023) — matches whole outcome distributions, needs micro-level data.
> - This notebook's DSC = **Demeaned** synthetic control (Doudchenko & Imbens 2016; Ferman & Pinto 2021) = `TSSC(method="MSCa")`.
>
> Same three letters, different estimators. Importing the wrong one raises no error; it just answers a different question.

In [ ]:
# Colab setup. Takes about a minute the first time.
%pip install -q "git+https://github.com/jgreathouse9/mlsynth.git" 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mlsynth
from mlsynth import FDID, MASC, SDID, TSSC, VanillaSC
from mlsynth.exceptions import MlsynthConfigError, MlsynthDataError

print("mlsynth", mlsynth.__version__)
print("estimators exported:", len(mlsynth.__all__))

## 1. The data and the mlsynth contract

Every estimator in the library wants the same five things: a **long-format** DataFrame with one row per unit-period, plus the names of the outcome, treatment, unit and time columns.

`treat` is a 0/1 indicator that is 1 for the treated unit **in post-treatment periods** and 0 everywhere else — including for the treated unit *before* treatment. `mlsynth` reads both the donor pool and the treatment date off that one column.

In [ ]:
# Loads over HTTPS on Colab; falls back to the sibling copy in the R post,
# and to a local file if you downloaded the Quarto bundle.
URLS = [
    "https://raw.githubusercontent.com/cmg777/starter-academic-v501/"
    "master/content/post/python_sc_dsc_sdid/brexit_analysis.csv",
    "https://raw.githubusercontent.com/cmg777/starter-academic-v501/"
    "master/content/post/r_sc_dsc_sdid/brexit_analysis.csv",
    "brexit_analysis.csv",
]
for src in URLS:
    try:
        panel = pd.read_csv(src)
        print(f"loaded from {src}")
        break
    except Exception:
        continue
else:
    raise RuntimeError("could not load brexit_analysis.csv from any source")

print(f"{panel.shape[0]} rows x {panel.shape[1]} columns")
print(f"{panel.country.nunique()} countries, quarters t = {panel.t.min()}..{panel.t.max()}")
panel[["country", "quarter_label", "t", "log_rgdp", "treated"]].head()

In [ ]:
T0 = 86                                    # pre-treatment quarters, through 2016Q2
EVAL = {"2018Q4": 96, "2019Q4": 100}       # evaluation quarters, as values of `t`
TREATED = "United Kingdom"

Y = panel.pivot(index="t", columns="country", values="log_rgdp").sort_index()
DONORS = [c for c in Y.columns if c != TREATED]
QLAB = panel[panel.country == TREATED].sort_values("t")["quarter_label"].to_numpy()
QDEC = panel[panel.country == TREATED].sort_values("t")["date_dec"].to_numpy()

print(f"{len(DONORS)} donors, T0 = {T0} (last pre-treatment quarter {QLAB[T0-1]})")

### The one trick that makes everything else simple

Every `mlsynth` estimator reports an ATT averaged over **all** post-treatment periods. We want the shortfall at two specific quarters.

So: keep the 86 pre-treatment quarters **plus the single quarter of interest**, renumber time to run 1..87, and "the average over all post periods" becomes an average over one period. A bare `.fit()` then returns exactly the number we want — no post-estimation arithmetic anywhere.

In [ ]:
def window(post, pre=T0):
    '''`pre` pre-treatment quarters + the given post quarter(s), renumbered 1..pre+1.'''
    post = [post] if np.isscalar(post) else list(post)
    sub = panel[(panel.t <= pre) | (panel.t.isin(post))].copy()
    sub["tt"] = sub.groupby("country")["t"].rank(method="dense").astype(int)
    sub["treat"] = ((sub.country == TREATED) & (sub.tt > pre)).astype(int)
    return sub


def cfg(post, pre=T0, **extra):
    '''The five fields every mlsynth estimator wants, plus estimator-specific ones.'''
    return dict(df=window(post, pre), outcome="log_rgdp", treat="treat",
                unitid="country", time="tt", display_graphs=False, **extra)


def pct(att):
    '''mlsynth reports treated minus counterfactual. Flip into a % GDP shortfall.'''
    return -100.0 * att


def tssc_att(res, variant="MSCa", n_post=1):
    '''Full-precision ATT for a TSSC variant.

    TSSC rounds its scalar summaries (.att comes back as exactly -0.03) but the
    gap and counterfactual SERIES are full precision. See section 5.
    '''
    gap = np.asarray(res.variants[variant].gap, float).ravel()
    return float(gap[-n_post:].mean())

### Configs are validated

The configs are Pydantic v2 models with `extra = "forbid"`, so a typo raises instead of being silently ignored. There are four exception types, and they tell you which stage failed: `MlsynthConfigError` (you wrote the config wrong), `MlsynthDataError` (the DataFrame breaks the contract), `MlsynthEstimationError` (the optimiser gave up) and `MlsynthPlottingError`.

In [ ]:
base = cfg(EVAL["2018Q4"])

for bad, why in [
    (dict(base, backendd="outcome-only"), "misspelled keyword"),
    (dict(base, outcome="gdp_log"),       "column not in the DataFrame"),
]:
    try:
        VanillaSC(bad).fit()
    except (MlsynthConfigError, MlsynthDataError) as exc:
        print(f"{why:<28s} -> {type(exc).__name__}: {str(exc).splitlines()[0][:70]}")

## 2. Anatomy of a fit

The call pattern never changes: build a config, construct the estimator, call `.fit()`, read the result.

The result carries six sub-models (`effects`, `fit_diagnostics`, `time_series`, `weights`, `inference`, `method_details`) plus seven flat accessors that work on **every** effect estimator in the library:

`.att` · `.att_ci` · `.counterfactual` · `.gap` · `.donor_weights` · `.weight_vector` · `.pre_rmse`

In [ ]:
res = VanillaSC(cfg(EVAL["2018Q4"], inference=False)).fit()

print(f"type(result)                 {type(res).__name__}")
print(f"result.att                   {res.att:+.6f}")
print(f"result.pre_rmse              {res.pre_rmse:.6f}")
print(f"result.fit_diagnostics.r_squared_pre  {res.fit_diagnostics.r_squared_pre:.6f}")
print(f"result.method_details.method_name     {res.method_details.method_name}")
print(f"len(result.donor_weights)    {len(res.donor_weights)}   # only NON-ZERO donors")
print(f"result.counterfactual.shape  {np.shape(res.counterfactual)}")

### Plotting comes free

Every effect result carries `.plot()`, driven by a `PlotConfig` captured at fit time. This is the package's own output with no styling from us.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
res.plot(kind="counterfactual", ax=axes[0])
res.plot(kind="gap", ax=axes[1])
plt.tight_layout()
plt.show()

## 3. Rung 0 — Difference-in-differences

`mlsynth` has no standalone DiD class. The plain two-way estimator comes free inside `FDID` (Forward DiD), which fits its own estimator and the textbook benchmark side by side and exposes them as `.fdid` and `.did`.

The uniform 1/23 donor weights are the giveaway that nothing was fitted.

In [ ]:
fdid_res = {k: FDID(cfg(e)).fit() for k, e in EVAL.items()}
did = {k: r.did for k, r in fdid_res.items()}

print(f"DiD  2018Q4 {pct(did['2018Q4'].att):.2f}%   2019Q4 {pct(did['2019Q4'].att):.2f}%")
print(f"SE (analytic) {100 * did['2018Q4'].att_se:.2f}")

wd = did["2018Q4"].donor_weights
print(f"donor weights: {len(wd)} donors, all equal to {next(iter(wd.values())):.6f} = 1/{len(wd)}")
print(f"pre-treatment RMSE {did['2018Q4'].pre_rmse:.5f}, R^2 {did['2018Q4'].r_squared:.4f}")

# A free seventh rung: Forward DiD selects a subset of donors, then runs DiD on them.
f18 = fdid_res["2018Q4"].fdid
print(f"\nForward DiD selects {len(f18.selected_names)} donors: {', '.join(f18.selected_names)}")
print(f"Forward DiD  2018Q4 {pct(f18.att):.2f}%   pre-RMSE {f18.pre_rmse:.5f}")

## 4. Rung 1 — Synthetic control

DiD's complaint is that the donor average does not look like the UK. Synthetic control fits the unit weights on the simplex:

$$\hat\omega = \arg\min_{\omega} \sum_{t=1}^{T_0} \Big( Y_{\text{UK},t} - \sum_j \omega_j Y_{j,t} \Big)^2,
\qquad \omega_j \ge 0, \ \sum_j \omega_j = 1.$$

In [ ]:
sc = {k: VanillaSC(cfg(e, inference=False)).fit() for k, e in EVAL.items()}
s18 = sc["2018Q4"]

print(f"SC  2018Q4 {pct(s18.att):.2f}%   2019Q4 {pct(sc['2019Q4'].att):.2f}%")
print(f"backend chosen by 'auto': {s18.method_details.method_name}")
print(f"pre-RMSE {s18.pre_rmse:.6f}   R^2 {s18.fit_diagnostics.r_squared_pre:.5f}\n")

ss = s18.weights.summary_stats
print(f"weights: {ss['n_nonzero']} nonzero, sum {ss['sum_of_weights']:.6f}, {ss['constraint']}")
for c, w in sorted(s18.donor_weights.items(), key=lambda kv: -kv[1]):
    if w > 0.01:
        print(f"    {c:<16s} {w:.4f}")

### One estimator, four solvers

Plain synthetic control is a single mathematical object, but `mlsynth` can reach it four different ways — and R's `synthdid` reaches it a fifth.

The SC objective on this panel has a condition number of about $7.5 \times 10^5$: a long, narrow, nearly flat valley of near-optimal weight vectors. Any optimiser has to decide when to stop walking down it.

In [ ]:
for label, klass, kw in [
    ("VanillaSC, backend='auto'",         VanillaSC, dict(inference=False)),
    ("VanillaSC, backend='outcome-only'", VanillaSC, dict(backend="outcome-only", inference=False)),
    ("VanillaSC, w_constr='simplex'",     VanillaSC, dict(w_constr="simplex", inference=False)),
    ("TSSC, method='SC'",                 TSSC,      dict(method="SC", inference=False)),
]:
    r = klass(cfg(EVAL["2018Q4"], **kw)).fit()
    att = tssc_att(r, "SC") if klass is TSSC else r.effects.att
    print(f"{label:<36s} 2018Q4 {pct(att):.3f}")

print(f"{'R synthdid (Frank-Wolfe, published)':<36s} 2018Q4 3.060")

Four independent code paths inside `mlsynth` agree to three decimals at **3.039%**. R's `synthdid` stops at **3.06%**.

Neither is buggy. `synthdid` walks the valley with Frank-Wolfe on a capped iteration budget; `mlsynth` hands the identical problem to a convex solver that runs it to optimality. **A synthetic control estimate carries its solver's fingerprint**, and a second-decimal disagreement between implementations is normal rather than alarming.

## 5. Rung 2 — Demeaned synthetic control

SC insists the blend match the UK's *level*. DSC adds one free parameter — a constant offset — so the blend has to match the shape but not the level. In `mlsynth` this is the `MSCa` variant of `TSSC`.

**A precision trap.** `TSSC` rounds every scalar it reports. The series are full precision; `.att`, `.rmse_pre` and the donor weights are not.

In [ ]:
dsc = {k: TSSC(cfg(e, method="MSCa", inference=False)).fit() for k, e in EVAL.items()}
d18 = dsc["2018Q4"].variants["MSCa"]

print(f"DSC  2018Q4 {pct(tssc_att(dsc['2018Q4'])):.2f}%   2019Q4 {pct(tssc_att(dsc['2019Q4'])):.2f}%")
print(f"MSCa intercept: {d18.intercept:+.6f} log points = {100 * d18.intercept:+.3f}% of GDP\n")

print("The rounding trap:")
print(f"  variants['MSCa'].att      {d18.att!r}   -> {pct(d18.att):.4f}%")
print(f"  gap[-1] (full precision)  {float(np.asarray(d18.gap, float)[-1]):.16f}")
print(f"                            -> {pct(tssc_att(dsc['2018Q4'])):.4f}%")
print(f"  variants['MSCa'].rmse_pre {d18.rmse_pre!r}")

### The four variants and the Step-1 selection

Left to itself `TSSC` fits all four variants and runs a subsampling procedure to pick one. `method=` forces a single variant and skips the selection — a speed-up of roughly 1700x, which is what makes the placebo loop in section 9 tractable.

Note that `inference=False` **requires** `method` to be set: asking for no inference without naming a variant raises, because there would be nothing to select with.

In [ ]:
t_all = TSSC(cfg(EVAL["2018Q4"], draws=500, seed=20260801)).fit()

print(f"TSSC recommends: {t_all.recommended_method}\n")
for m in ("SC", "MSCa", "MSCb", "MSCc"):
    v = t_all.variants[m]
    ic = "none" if v.intercept is None else f"{v.intercept:+.5f}"
    print(f"  {m:<5s} loss {pct(tssc_att(t_all, m)):5.3f}%   "
          f"(.att reports {v.att:+.5f})   intercept {ic}")

if t_all.selection is not None:
    print()
    for name, test in t_all.selection.tests.items():
        print(f"  test '{name}': stat {test.statistic:+.5f}  rejected {test.rejected}")

## 6. Rung 3 — Synthetic difference-in-differences

DSC still treats all 86 pre-treatment quarters as equally informative. SDID fits the **time weights** too.

**The one setting that carries the result:** `zeta` is a ridge penalty on the unit weights, and it is on by default. Every implementation in every language penalises by default — R's `synthdid` needs `zeta.omega = 0`, Stata's `sdid` needs `zeta_omega(0)`.

In [ ]:
sdid = {k: SDID(cfg(e, zeta=0.0, vce="placebo", B=500, seed=20260801)).fit()
        for k, e in EVAL.items()}
s = sdid["2018Q4"]

print(f"SDID  2018Q4 {pct(s.effects.att):.2f}%   2019Q4 {pct(sdid['2019Q4'].effects.att):.2f}%")

inf = s.inference_detail
print(f"ATT {inf.att:+.5f}, SE {inf.se:.5f}, CI [{inf.ci[0]:+.5f}, {inf.ci[1]:+.5f}]")
print(f"p = {inf.p_value:.4f}, method '{inf.method}', n_placebo {inf.n_placebo}\n")

default = SDID(cfg(EVAL["2018Q4"], vce="noinference")).fit()
print(f"zeta left at its default: {pct(default.effects.att):.2f}%")
print(f"zeta = 0.0:               {pct(s.effects.att):.2f}%   <- the paper's specification")

### The time weights are not where you would first look

Unit weights are `result.donor_weights`. SDID's time weights live on the **cohort** object: `result.cohorts[a].time_weights`.

In [ ]:
coh = list(sdid["2018Q4"].cohorts.values())[0]
lam = np.asarray(coh.time_weights, float)

print(f"cohorts: {list(sdid['2018Q4'].cohorts)}  (n_treated={coh.n_treated}, n_post={coh.n_post})")
print(f"lambda: {len(lam)} weights summing to {lam.sum():.6f}\n")
for i in np.where(lam > 1e-4)[0]:
    print(f"    {QLAB[i]:<8s} {lam[i]:.4f}")

print(f"\nDiD would use the uniform weight 1/{T0} = {1/T0:.4f} on every quarter.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axhline(1 / T0, color="#e8b04b", ls="--", lw=1.0,
           label="uniform (the two-way fixed-effects correction)")
ax.vlines(QDEC[:T0], 0, lam, color="#6a9bcc", lw=0.8)
ax.plot(QDEC[:T0], lam, "o", color="#6a9bcc", ms=3.5)
big = lam > 0.02
ax.plot(QDEC[:T0][big], lam[big], "o", color="#d97757", ms=9)
for i in np.where(big)[0]:
    ax.annotate(f"{QLAB[i]}: {lam[i]:.3f}", xy=(QDEC[i], lam[i]),
                xytext=(-12, -4) if lam[i] > 0.5 else (0, 12),
                textcoords="offset points", color="#d97757",
                ha="right" if lam[i] > 0.5 else "center")
ax.set_xlabel("year"); ax.set_ylabel(r"time weight $\lambda_s$")
ax.set_title("SDID's time weights collapse onto the last pre-treatment quarter")
ax.legend()
plt.show()

Log real GDP behaves close to a random walk. If the outcome is a random walk, the best predictor of next quarter is *this* quarter, and the 85 quarters before it add noise rather than information. Read the collapse as SDID correctly discovering that most of the pre-treatment history is uninformative — not as a bug.

### The event study

`mlsynth` aggregates the Ciccia (2024) event-study estimator alongside the headline ATT, which the R packages on this ladder do not. The flat pre-treatment path is a falsification test the single ATT number cannot give you.

In [ ]:
full = cfg(list(range(T0 + 1, len(QLAB) + 1)))
full_sdid = SDID(dict(full, zeta=0.0, vce="placebo", B=200, seed=20260801)).fit()

es = full_sdid.event_study
et, tau, ci = (np.asarray(es.event_times, float), np.asarray(es.tau, float),
               np.asarray(es.ci, float))

print(f"event times {et.min():.0f}..{et.max():.0f}")
print(f"post-treatment tau mean {tau[et > 0].mean():+.5f}")
print(f"pre-treatment tau mean  {tau[(et < 0) & (et >= -20)].mean():+.5f}   (should be ~0)")

keep = (et >= -20) & (et <= 18)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axhline(0, color="grey", lw=0.8); ax.axvline(0, color="black", ls="--", lw=1.0)
ax.fill_between(et[keep], ci[keep, 0], ci[keep, 1], color="#6a9bcc", alpha=0.25)
ax.plot(et[keep], tau[keep], "o-", color="#d97757", lw=1.7, ms=4)
ax.set_xlabel("quarters since the referendum"); ax.set_ylabel("effect on log real GDP")
ax.set_title("SDID event study, straight from result.event_study")
plt.show()

### Three flavours of SDID

The time weights have to be fitted against *something* in the post period, and there are three natural choices:

| Variant | $\lambda$ is fitted to predict |
|---|---|
| (i) | the first treated quarter, 2016Q3 |
| (ii) | the average of 2016Q3 through the evaluation date |
| (iii) | the evaluation quarter alone |

Only (iii) falls out of a bare `.fit()`. The other two need the weights the result already hands you, applied at a different quarter:

$$\hat\tau_t = \Big( Y_{\text{UK},t} - \sum_j \hat\omega_j Y_{j,t} \Big)
             - \sum_{s \le T_0} \hat\lambda_s \Big( Y_{\text{UK},s} - \sum_j \hat\omega_j Y_{j,s} \Big)$$

In [ ]:
def sdid_weights(post, pre=T0):
    '''Fit SDID on a given post window; return (omega dict, lambda array).'''
    res = SDID(cfg(post, pre, zeta=0.0, vce="noinference")).fit()
    return res.donor_weights, np.asarray(list(res.cohorts.values())[0].time_weights, float)


def sdid_loss(w, lam, t, pre=T0):
    '''Apply an (omega, lambda) pair at an arbitrary quarter.'''
    wv = np.array([w[c] for c in DONORS])
    gap = lambda s: float(Y.loc[s, TREATED] - Y.loc[s, DONORS].to_numpy() @ wv)
    bias = float(sum(lam[s - 1] * gap(s) for s in range(1, pre + 1)))
    return -100.0 * (gap(t) - bias)


w_i,   lam_i   = sdid_weights(T0 + 1)                          # (i)
w_ii,  lam_ii  = sdid_weights(range(T0 + 1, EVAL["2018Q4"] + 1))  # (ii)
w_iii, lam_iii = sdid_weights(EVAL["2018Q4"])                  # (iii)

for name, (w, lam) in {"SDID (i)": (w_i, lam_i), "SDID (ii)": (w_ii, lam_ii),
                       "SDID (iii)": (w_iii, lam_iii)}.items():
    print(f"{name:<11s} 2018Q4 {sdid_loss(w, lam, EVAL['2018Q4']):.3f}")

The three land within **0.03 percentage points** of each other, because all three put essentially all their time weight on the same last pre-treatment quarter.

## 7. Rung 4 — MASC

MASC forms a convex combination of synthetic control and $m$-nearest-neighbour matching, choosing $m$ and $\phi$ jointly by rolling-origin cross-validation:

$$\hat{Y}^{\text{MASC}} = \phi \cdot \hat{Y}^{\text{match}}_m + (1-\phi) \cdot \hat{Y}^{\text{SC}}$$

**The argument that decides the answer** is `set_f`. Its alternative `min_preperiods` defaults to $\lceil T_0/2 \rceil$, which is not the fold set the paper uses — and the difference is 0.47 percentage points.

In [ ]:
M_GRID = list(range(1, 11))
SET_F = list(range(6, T0 + 1))

masc = {k: MASC(cfg(e, m_grid=M_GRID, set_f=SET_F)).fit() for k, e in EVAL.items()}
m18 = masc["2018Q4"]

print(f"MASC  2018Q4 {pct(m18.att):.2f}%   2019Q4 {pct(masc['2019Q4'].att):.2f}%")
# The two tuned dials live in weights.summary_stats, NOT method_details.parameters_used
print(f"weights.summary_stats: {m18.weights.summary_stats}\n")

for label, kw in [
    ("set_f=range(6, 87)  [the paper]",  dict(m_grid=M_GRID, set_f=SET_F)),
    ("min_preperiods=None [default]",    dict(m_grid=M_GRID)),
    ("min_preperiods=43   [ceil(T0/2)]", dict(m_grid=M_GRID, min_preperiods=43)),
]:
    vals = [pct(MASC(cfg(e, **kw)).fit().att) for e in EVAL.values()]
    print(f"{label:<34s} 2018Q4 {vals[0]:.3f}   2019Q4 {vals[1]:.3f}")

## 8. Rung 5 — Augmented synthetic control

Every rung so far assumes the treated unit lies inside the convex hull of the donors. ASCM fits a ridge regression to whatever pre-treatment imbalance is left and corrects for it, using negative weights if it needs them. One keyword on the class you already used.

In [ ]:
ascm = {k: VanillaSC(cfg(e, augment="ridge", inference=False)).fit() for k, e in EVAL.items()}
a18 = ascm["2018Q4"]
aw = np.array(list(a18.donor_weights.values()))

print(f"ASCM  2018Q4 {pct(a18.att):.2f}%   2019Q4 {pct(ascm['2019Q4'].att):.2f}%")
print(f"weights: sum {aw.sum():.5f}, {int((aw < -1e-6).sum())} negative, "
      f"min {aw.min():+.4f}, max {aw.max():+.4f}")
print(f"pre-RMSE {a18.pre_rmse:.6f} vs SC's {s18.pre_rmse:.6f}")

The negative weights are all tiny and the pre-treatment RMSE barely improves: there was almost no imbalance left to fix, because the UK sits comfortably inside the convex hull of 23 OECD economies. Augmentation earns its keep when the treated unit is extreme.

## 9. The whole ladder, side by side

In [ ]:
LADDER = [
    ("DiD",  "FDID(...).fit().did",            pct(did["2018Q4"].att),          pct(did["2019Q4"].att),          4.98, np.nan),
    ("SC",   'VanillaSC(...)',                 pct(sc["2018Q4"].att),           pct(sc["2019Q4"].att),           3.06, 3.06),
    ("DSC",  'TSSC(..., method="MSCa")',       pct(tssc_att(dsc["2018Q4"])),    pct(tssc_att(dsc["2019Q4"])),    2.98, 2.98),
    ("SDID", "SDID(..., zeta=0.0)",            pct(sdid["2018Q4"].effects.att), pct(sdid["2019Q4"].effects.att), 2.79, 2.79),
    ("MASC", "MASC(..., set_f=range(6, 87))",  pct(masc["2018Q4"].att),         pct(masc["2019Q4"].att),         2.73, 2.73),
    ("ASCM", 'VanillaSC(..., augment="ridge")', pct(ascm["2018Q4"].att),        pct(ascm["2019Q4"].att),         3.04, 3.04),
]
ladder = pd.DataFrame(LADDER, columns=["method", "command", "loss_2018Q4",
                                       "loss_2019Q4", "R post", "Paper"])
ladder.round(2)

Four of six rungs agree with both the R edition and the published paper to two decimals. The two that do not — SC and SDID — differ in the same direction and for the same reason: the solver, not the estimator.

**Every rung puts the cost above the 2.4% previously published for this dataset.**

## 10. Do covariates help? Three meanings of "control for"

`SDIDConfig.covariates` is **not a list**. It is a dictionary keyed by method, because the literature contains three different answers:

| Key | Method | What it does |
|---|---|---|
| `"adjust"` | Kranz (2022) | residualise the *outcome* on covariates first |
| `"match"` | de Brabander et al. (2025) | put the covariates *inside the unit-weight problem* |
| `"optimized"` | Arkhangelsky et al. (2021) | estimate weights and coefficients *jointly* |

In [ ]:
COVARIATES = ["cons_share", "inv_share", "exp_share", "imp_share",
              "labprod_growth", "emp_pop"]

print(f"outcomes only               2018Q4 {pct(sdid['2018Q4'].effects.att):5.2f}")
for meth in ("adjust", "match", "optimized"):
    kw = dict(zeta=0.0, vce="noinference", covariates={meth: COVARIATES})
    if meth == "match":
        kw["match_pre_periods"] = "last"
    vals = [pct(SDID(cfg(e, **kw)).fit().effects.att) for e in EVAL.values()]
    print(f"covariates={{'{meth}': ...}}{'':<{14 - len(meth)}} "
          f"2018Q4 {vals[0]:5.2f}   2019Q4 {vals[1]:5.2f}")

The three routes disagree by nearly **two percentage points** — more than five times the spread across the entire outcomes-only ladder. Adding covariates here does not refine the answer; it replaces one well-identified number with three poorly-identified ones.

### Why: the predictor weights are not identified

`VanillaSC`'s covariate route is the Abadie-Diamond-Hainmueller bilevel program — an outer loop over predictor weights $V$, an inner loop for donor weights $\omega$. Those predictor weights are *generically not identified*, and there is a one-keyword test for it: fit the identical model twice with the same seed and the same data, changing only how long the search runs. If $V$ were well identified, the budget would not matter.

In [ ]:
for label, budget in [("default (maxiter=300, popsize=15)", {}),
                      ("reduced (maxiter=120, popsize=12)",
                       dict(mscmt_maxiter=120, mscmt_popsize=12))]:
    r = VanillaSC(cfg(EVAL["2018Q4"], covariates=COVARIATES, backend="mscmt",
                      canonical_v="min.loss.w", seed=20260801,
                      inference=False, **budget)).fit()
    ss = r.weights.summary_stats
    print(f"{label:<36s} {pct(r.effects.att):5.2f}%   "
          f"pre-RMSE {r.fit_diagnostics.rmse_pre:.6f}   "
          f"v_agreement {ss['v_agreement']:.5f}")
    top = sorted(ss["predictor_weights"].items(), key=lambda kv: -abs(kv[1]))[:4]
    print("    " + ", ".join(f"{k} {v:.3f}" for k, v in top))

print(f"\noutcomes-only pre-RMSE for comparison: {s18.pre_rmse:.6f}")

Same estimator, same seed, same data. Shortening the search moves the estimate **and** moves the predictor weights from a corner solution on the import share to a spread across four predictors. Those are different economic stories about what makes a country comparable to the UK.

The pre-treatment RMSE also **rises** when covariates are added — the optimiser now spends its effort matching predictor means instead of the outcome path. With 86 quarters of the outcome itself, six covariate means are not adding information; they are adding a poorly identified optimisation problem.

## 11. Inference

`VanillaSC` exposes nine inference methods behind one `inference=` field.

In [ ]:
rows = []
for meth in ("placebo", "scpi", "lto", "conformal", "ttest"):
    r = VanillaSC(cfg(EVAL["2018Q4"], inference=meth, alpha=0.05)).fit()
    inf = r.inference
    rows.append(dict(method=meth, p_value=inf.p_value,
                     ci_lower=inf.ci_lower, ci_upper=inf.ci_upper,
                     reported_as=inf.method))
pd.DataFrame(rows)

All five reject at the 5% level. Read them as orders of magnitude, not digits: they are not independent tests — they share the point estimate and the donor pool, and differ only in how they build a reference distribution from 23 donors. Note that SDID's own placebo inference above gave p = 0.20 with an interval containing zero.

**Placebo in space** is the most interpretable of the five: give every donor the treatment in turn and see where the UK ranks, using the ratio of post- to pre-treatment RMSPE.

In [ ]:
rows = []
for country in [TREATED] + DONORS:
    sub = panel.copy()
    sub["tt"] = sub.groupby("country")["t"].rank(method="dense").astype(int)
    sub["treat"] = ((sub.country == country) & (sub.tt > T0)).astype(int)
    r = VanillaSC(dict(df=sub, outcome="log_rgdp", treat="treat", unitid="country",
                       time="tt", display_graphs=False, inference=False)).fit()
    gap = np.asarray(r.gap, float).ravel()
    pre, post = np.sqrt(np.mean(gap[:T0] ** 2)), np.sqrt(np.mean(gap[T0:] ** 2))
    rows.append(dict(country=country, rmspe_pre=pre, rmspe_post=post,
                     ratio=post / pre, gap=gap))

space = pd.DataFrame([{k: v for k, v in r.items() if k != "gap"} for r in rows])
space = space.sort_values("ratio", ascending=False).reset_index(drop=True)
uk_rank = int(space.index[space.country == TREATED][0]) + 1
print(f"UK ranks {uk_rank} of {len(space)}  ->  permutation p = {uk_rank/len(space):.3f}")
print(f"(with {len(space)} units the smallest attainable p is {1/len(space):.3f})\n")
space.head(6).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for r in rows:
    if r["country"] != TREATED:
        ax.plot(QDEC[:len(r["gap"])], r["gap"], color="#54618a", lw=0.8, alpha=0.6)
uk = next(r for r in rows if r["country"] == TREATED)
ax.plot(QDEC[:len(uk["gap"])], uk["gap"], color="#d97757", lw=2.4, label="United Kingdom")
ax.plot([], [], color="#54618a", lw=1.0, label="23 placebo donors")
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(QDEC[T0], color="black", ls="--", lw=1.0)
ax.set_xlabel("year"); ax.set_ylabel("gap (log points)")
ax.set_title("Placebo in space: give every donor the treatment in turn")
ax.legend(loc="lower left")
plt.show()

## 12. Summary

- The referendum cost the UK **2.7–3.1% of GDP by end-2018** and **3.8–4.2% by end-2019** on every rung, against 2.4% previously published.
- `mlsynth` puts all six rungs behind one interface: `Estimator({"df":…, "outcome":…, "treat":…, "unitid":…, "time":…}).fit()`.
- **Three defaults matter more than the estimator choice.** `zeta` moves SDID by 0.13 points, `set_f` moves MASC by 0.47, and the three covariate methods disagree by 1.76. The ladder's own spread, excluding DiD, is 0.31.
- **`TSSC` rounds `.att`.** Read the gap series instead.
- **`mlsynth.DSC` is not this notebook's DSC.** Class names are mnemonics, not definitions.
- **A solver leaves a fingerprint.** Four convex paths inside `mlsynth` land on 3.039%; R's Frank-Wolfe stops at 3.06%.

### Where to go next

The [full post](https://carlos-mendez.org/post/python_sc_dsc_sdid/) adds the in-sample placebo tournament over twenty artificial treatment dates, the robustness zoo, and a map of the wider `mlsynth` catalogue. [The R edition](https://carlos-mendez.org/post/r_sc_dsc_sdid/) hand-codes every estimator before calling its package, and is the place to go for the derivations.

---

*AI tools (Claude Code, Gemini, NotebookLM) were used to make the contents of this notebook more accessible to students. Nevertheless, the content may still have errors. Caution is needed when applying it to true research projects.*